In [17]:
from pymarc import MARCReader
import pandas as pd
import requests
from dotenv import load_dotenv
import os
from pinecone import Pinecone
import time
import datetime
import json
import glob
import numpy as np

In [18]:
# Load environment variables from .env file
load_dotenv()

# Get ISBN API key from environment variables
ISBN_API_KEY = os.getenv('ISBN_API_KEY')
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

In [19]:
today = datetime.datetime.now()

In [20]:
def clean_text(text):
    text = text.replace(' /', '')
    text = text.replace('.', '')
    return text

In [21]:
def clean_authors(authors):
    a = clean_text(authors)
    # Split on comma and reverse order
    if ',' in a:
        last_name, first_name = a.split(',', 1)
        return f"{first_name.strip()} {last_name.strip()}"
    return a

In [22]:
def extract_book_info(response_json):
    """
    Extract specific fields from ISBN API response and return as dictionary.
    
    Args:
        response_json (dict): The JSON response from the ISBN API
        
    Returns:
        dict: Dictionary containing extracted book information
    """
    extracted_info = {}
    
    # The response might have the book data nested under 'book' key
    book_data = response_json.get('book', response_json)
    
    extracted_info['_id'] = book_data.get('isbn', '')
    
    # Extract synopsis (might be under 'synopsis', 'overview', or 'description')
    extracted_info['synopsis'] = (
        book_data.get('synopsis') or 
        book_data.get('overview') or 
        book_data.get('description') or 
        ''
    )
    
    # Extract title_long
    extracted_info['title'] = book_data.get('title_long', book_data.get('title', ''))
    
    
    # Extract subjects (might be a list or string)
    subjects = book_data.get('subjects')
    if isinstance(subjects, list):
        extracted_info['subjects'] = subjects
    elif isinstance(subjects, str):
        extracted_info['subjects'] = [subjects]
    else:
        extracted_info['subjects'] = ''
    
    # Extract authors (might be a list or string)
    authors = book_data.get('authors')
    if isinstance(authors, list):
        extracted_info['authors'] = clean_authors(authors[0])
    elif isinstance(authors, str):
        extracted_info['authors'] = clean_authors(authors)
    else:
        extracted_info['authors'] = ''

    extracted_info['series'] = 'None'
    
    return extracted_info


In [23]:
def get_book_info(isbn):
    # https://isbndb.com/user/62794
    if isbn is None:
        return None

    # Clean the ISBN (remove any extra characters, spaces, etc.)
    clean_isbn = isbn.replace('-', '').replace(' ', '').strip()
    
    # API request to ISBN database
    url = f"https://api2.isbndb.com/book/{clean_isbn}"
    headers = {
        'User-Agent': 'python-requests/2.28.1',
        'Authorization': ISBN_API_KEY,  # Replace with your actual API key
        'Accept': '*/*'
    }
    
    
    try:
        start_time = time.time()
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            book_info = extract_book_info(response.json())
            end_time = time.time()
            if end_time - start_time > 1:
                return book_info
            else:
                time.sleep(1)
                return book_info
        else:
            print(f"Error Response: {response.text}")
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")


In [24]:
def check_none_values(record_dict):
    """
    Check each item in a single record dictionary for None values.
    Replaces None values with empty strings and prints what was changed.
    
    Args:
        record_dict (dict): A single record dictionary to check and modify
        
    Returns:
        dict: The modified dictionary with None values replaced by empty strings
    """
    none_count = 0
    
    try:
        for key, value in record_dict.items():
            if value is None or value == '' or value == 'null':
                record_dict[key] = '-999'  # Replace None with empty string
            none_count += 1
    except Exception as e:
        print(f"Error checking record: {e}")
        return None

    return record_dict

In [25]:
# Path to your MARC file
marc_file = 'ExportBibJob605495USMarc.001'

In [26]:
existing_records = {}

# Get all JSON files in the data folder
json_files = glob.glob('data/error_records_*.json')

# Load and combine all records, using dict to automatically handle duplicates
for file in json_files:
    try:
        file_records = json.load(open(file))
        # Convert list of records to dict keyed by _id
        for r in file_records:
            if r is None:
                continue
            else:
                existing_records[r['_id']] = r
    except Exception as e:
        print(f"Error loading {file}: {e}")

# Convert back to list for consistency with rest of code
existing_records = list(existing_records.values())
print(f"Loaded {len(existing_records)} unique records")

Loaded 0 unique records


In [27]:
books = []
isbn_errors = []
true_errors = 0
with open(marc_file, 'rb') as file:
    reader = MARCReader(file, to_unicode=True, force_utf8=True)
    for record in reader:
        try:
            try:
                isbn = record['020']['a'] if record['020'] else ''
            except:
                print("ISBN not found")

            authors = record['100']['a'] if record['100'] else ''
            title = record['245']['a'] if record['245'] else ''
            summary = record['520']['a'] if record['520'] else ''
            bib_id = record['001'].data if record['001'] else ''
            try:    
                series = record['490']['a'] if record['490'] else ''
            except:
                series = 'None'

            subjects = []
            for subject_field in record.get_fields('650'):
                if subject_field['a']:
                    subjects.append(clean_text(subject_field['a'] if subject_field['a'] else ''))

            books.append({
                '_id': isbn,
                'bib_id': bib_id,
                'authors': clean_authors(authors),
                'title': clean_text(title),
                'synopsis': summary,
                'series': series,
                'subjects': subjects
            })
                
        except:
            try:
                isbn_errors.append(record['020']['a'] if record['020'] else '')
            except:
                true_errors += 1

print("Number of books: ", len(books))
print("Number of extracted ISBNs: ", len(isbn_errors))
print("Number of true errors: ", true_errors)

ISBN not found
ISBN not found
ISBN not found
ISBN not found
ISBN not found
ISBN not found
ISBN not found
ISBN not found
Number of books:  10643
Number of extracted ISBNs:  631
Number of true errors:  7


In [28]:
# List to store records
records = []
num_errors = 0

for isbn in isbn_errors:
    # Skip if ISBN already exists in existing_records
    if isbn in [record.get('_id') for record in existing_records]:
        continue
    try:
        book_info = get_book_info(isbn)
        if book_info is None:
            continue
        else:
            book_info_cleaned = check_none_values(book_info)
            # Add the record to the list
            records.append(book_info_cleaned)
        
    except Exception as e:
        print(f"Error processing record {book_info.get('isbn', 'unknown')}: {e}")
        num_errors += 1

    if len(records) >= 4000:
        break

print(f"Number of records processed: {len(records)}")
print(f"Number of errors: {num_errors}")

Error processing record unknown: list index out of range
Error processing record unknown: list index out of range
Error processing record unknown: list index out of range
Error processing record unknown: list index out of range
Error Response: {"errorType":"string","errorMessage":"Not Found","trace":[]}
Error Response: {"errorType":"string","errorMessage":"Not Found","trace":[]}
Error Response: {"errorType":"string","errorMessage":"Not Found","trace":[]}
Error processing record unknown: list index out of range
Error processing record unknown: list index out of range
Error processing record unknown: list index out of range
Error processing record unknown: list index out of range
Error Response: {"errorType":"string","errorMessage":"Not Found","trace":[]}
Error processing record unknown: list index out of range
Number of records processed: 618
Number of errors: 9


In [29]:
# Create filename with date
filename = 'data/error_records_{}.json'.format(today.strftime('%Y-%m-%d'))

# Save records to JSON file
with open(filename, 'w') as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} records to {filename}")

Saved 618 records to data/error_records_2025-08-20.json


In [30]:
# Combine clean_records and books into one full list
all_records = records + books

print(f"Total number of records: {len(all_records)}")

Total number of records: 11261


In [31]:
clean_records = []
for r in all_records:
    new_r = check_none_values(r)
    if new_r is not None:
        clean_records.append(new_r)

print(len(clean_records))

11261


In [50]:
def create_text_embeddings(record):
    if record['series'] != 'None':
        text = """The book {title} was written by {authors}.  Here is a synopsis: {synopsis}.  It is in the {series} series and has the following subjects: {subjects}."""
        text = text.format(title=record['title'], authors=record['authors'], synopsis=record['synopsis'], series=record['series'], subjects=record['subjects'])
    else:
        text = """The book {title} was written by {authors}.  Here is a synopsis: {synopsis}.  It has the following subjects: {subjects}."""
        text = text.format(title=record['title'], authors=record['authors'], synopsis=record['synopsis'], subjects=record['subjects'])
    return text

In [51]:
# Apply create_text_embeddings to each record and add result as 'text' field
for record in clean_records:
    record['text'] = create_text_embeddings(record)

print(f"Added text embeddings to {len(clean_records)} records")

Added text embeddings to 11261 records


In [52]:
clean_records

[{'_id': '1422215334',
  'synopsis': 'Profiles a variety of animals with strange or surprising traits.',
  'title': "Extraordinary Animals (Ripley's Believe It or Not)",
  'subjects': ['ANIMALS_HABITS AND BEHAVIOR',
   'ANIMAL BEHAVIOR_JUVENILE LITERATURE'],
  'authors': "Ripley's Entertainment Inc",
  'series': 'None',
  'text': "The book Extraordinary Animals (Ripley's Believe It or Not) was written by Ripley's Entertainment Inc.  Here is a synopsis: Profiles a variety of animals with strange or surprising traits..  It has the following subjects: ['ANIMALS_HABITS AND BEHAVIOR', 'ANIMAL BEHAVIOR_JUVENILE LITERATURE']."},
 {'_id': '1422215385',
  'synopsis': 'Collects strange, interesting, and record-breaking trivia on vehicles, including a coin-covered Mini Cooper, racers on beer kegs, and a monster truck made of balloons.',
  'title': "Ripley's Believe It or Not!: Life in the Fast Lane",
  'subjects': ['VEHICLES',
   'VEHICLES_JUVENILE LITERATURE',
   'CURIOSITIES AND WONDERS',
   'C

In [55]:
def upsert_records_to_index(function_records, index_type):
    base_index_name = "whitman"

    if index_type == "sparse":
        index_name = f"{base_index_name}-sparse"
        model = "pinecone-sparse-english-v0"
    elif index_type == "dense":
        index_name = f"{base_index_name}-dense"
        model = "multilingual-e5-large" # "llama-text-embed-v2"

    if not pc.has_index(index_name):
        pc.create_index_for_model(
            name=index_name,
            cloud="aws",
            region="us-east-1",
            embed={
                "model": model,
                "field_map": {
                    "text": "text"
                }
            }
        )

    # Target the index
    index = pc.Index(index_name)

    problem_records = []
    count = 0
    limit = 50
    for i in range(0, len(function_records), limit):
        try:
            if count < limit:
                index.upsert_records(index_name, function_records[i:i+limit])
                count += 1
            else:
                count = 0
                print(f"Sleeping for 60 seconds")
                time.sleep(60)
        except Exception as e:
            print(f"Error upserting records: {e}")
            print(f"Count: {count}")
            problem_records.append(function_records[i:i+limit])

    problem_records_again = []
    count = 0
    for i in problem_records:
        try:
            if count < 5:
                index.upsert_records(index_name, i)
                count += 1
            else:
                count = 0
                print(f"Sleeping for 60 seconds")
                time.sleep(60)
        except Exception as e:
            print(f"Error upserting records: {e}")
            print(f"Count: {count}")
            problem_records_again.append(i)

    if len(problem_records_again) > 0:
        print(f"There are {len(problem_records_again)} records that need to be upserted again")
    else:
        print("All records have been upserted successfully")

    time.sleep(10)

    # View stats for the index
    stats = index.describe_index_stats()
    print(stats) 

In [56]:
upsert_records_to_index(clean_records, 'dense')
upsert_records_to_index(clean_records, 'sparse')

Sleeping for 60 seconds
Sleeping for 60 seconds
Sleeping for 60 seconds
Sleeping for 60 seconds
All records have been upserted successfully
{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'whitman-dense': {'vector_count': 11143}},
 'total_vector_count': 11143,
 'vector_type': 'dense'}
Sleeping for 60 seconds
Sleeping for 60 seconds
Sleeping for 60 seconds
Sleeping for 60 seconds
All records have been upserted successfully
{'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'whitman-sparse': {'vector_count': 11043}},
 'total_vector_count': 11043,
 'vector_type': 'sparse'}


In [57]:
# Delete existing indexes and recreate them with correct field mapping
# Run this once to fix the author search issue

def recreate_indexes():
    """Delete and recreate indexes with the corrected field mapping"""
    
    # Delete existing indexes
    for index_type in ["dense", "sparse"]:
        index_name = f"whitman-{index_type}"
        if pc.has_index(index_name):
            print(f"Deleting existing index: {index_name}")
            pc.delete_index(index_name)
            time.sleep(30)  # Wait for deletion to complete
    
    print("Recreating indexes with corrected field mapping...")
    # Recreate indexes with the fixed field mapping
    upsert_records_to_index(clean_records, 'dense')
    upsert_records_to_index(clean_records, 'sparse')
    print("Indexes recreated successfully!")

# Uncomment the line below to run the recreation process
#recreate_indexes()


### Testing

In [58]:
query = "what is a book that has to do with the loch ness monster?"

index_name = "whitman-dense"
index = pc.Index(index_name)
# Search the dense index and rerank results
reranked_results = index.search(
    namespace=index_name,
    query={
        "top_k": 3,
        "inputs": {
            'text': query
        }
    } 
)

# Print the reranked results
for hit in reranked_results['result']['hits']:
    hit_dict = hit.to_dict()
    print(hit_dict)

{'_id': '1640263640', '_score': 0.8867923617362976, 'fields': {'authors': 'Ken Karst', 'bib_id': '149755', 'series': 'None', 'subjects': ['Loch Ness monster'], 'synopsis': 'An in-depth study of the Loch Ness Monster, examining legends, popular reports, and scientific evidence that supports or refutes the existence of the mysterious phenomenon.', 'text': "The book Loch Ness monster was written by Ken Karst.  Here is a synopsis: An in-depth study of the Loch Ness Monster, examining legends, popular reports, and scientific evidence that supports or refutes the existence of the mysterious phenomenon..  It has the following subjects: ['Loch Ness monster'].", 'title': 'Loch Ness monster'}}
{'_id': '9781098242367', '_score': 0.8786979913711548, 'fields': {'authors': 'Elizabeth, Andrews', 'bib_id': '158981', 'series': 'Creatures of legend', 'subjects': ['Loch Ness monster', 'Monsters', 'Animals, Mythical', 'Mythical animals', 'Legends'], 'synopsis': '"This title explores the history of the Loc

In [59]:
query = "What are more books by Jeff Kinney?"
index_name = "whitman-sparse"
index = pc.Index(index_name)
# Search the dense index and rerank results
reranked_results = index.search(
    namespace=index_name,
    query={
        "top_k": 3,
        "inputs": {
            'text': query
        }
    } 
)

# Print the reranked results
for hit in reranked_results['result']['hits']:
    hit_dict = hit.to_dict()
    print(hit_dict)


{'_id': '9781432882655', '_score': 11.091064453125, 'fields': {'authors': 'Jeff Kinney', 'bib_id': '147612', 'series': 'Thorndike Press large print middle reader', 'subjects': ['Heroes', 'Friendship', 'Adventure stories', 'Humorous stories', 'Large type books', 'Adventure and adventurers', 'Fantasy'], 'synopsis': '"From the imagination of Rowley Jefferson comes an adventure of epic proportions. Join Roland and his best friend, Garg the Barbarian, as they leave the safety of their village and embark on a quest to save Roland\'s mom from the White Warlock"--Back cover.', 'text': 'The book Rowley Jefferson\'s Awesome friendly adventure was written by Jeff Kinney.  Here is a synopsis: "From the imagination of Rowley Jefferson comes an adventure of epic proportions. Join Roland and his best friend, Garg the Barbarian, as they leave the safety of their village and embark on a quest to save Roland\'s mom from the White Warlock"--Back cover..  It is in the Thorndike Press large print middle re

In [60]:
query = "What are more books by Jeff Kinney?"
index_name = "whitman-dense"
index = pc.Index(index_name)
# Search the dense index and rerank results
reranked_results = index.search(
    namespace=index_name,
    query={
        "top_k": 3,
        "inputs": {
            'text': query
        }
    } 
)

# Print the reranked results
for hit in reranked_results['result']['hits']:
    hit_dict = hit.to_dict()
    print(hit_dict)


{'_id': '9781419756979', '_score': 0.8236668109893799, 'fields': {'authors': 'Jeff Kinney', 'bib_id': '152707', 'series': 'None', 'subjects': ['Friendship', 'Humorous stories', 'Adventure stories', 'Adventure and adventurers', 'Supernatural', 'Ghosts'], 'synopsis': 'Presents a collection of fourteen humorous spooky stories from Greg Heffley\'s best friend Roland "Rowley" Jefferson.', 'text': 'The book Rowley Jefferson\'s awesome friendly spooky stories was written by Jeff Kinney.  Here is a synopsis: Presents a collection of fourteen humorous spooky stories from Greg Heffley\'s best friend Roland "Rowley" Jefferson..  It has the following subjects: [\'Friendship\', \'Humorous stories\', \'Adventure stories\', \'Adventure and adventurers\', \'Supernatural\', \'Ghosts\'].', 'title': "Rowley Jefferson's awesome friendly spooky stories"}}
{'_id': '9781419741913', '_score': 0.822912871837616, 'fields': {'authors': 'Jeff Kinney', 'bib_id': '148318', 'series': 'Diary of a wimpy kid ;', 'subje

In [61]:
query = "Diary of a Wimpy Kid"
index_name = "whitman-sparse"
index = pc.Index(index_name)
# Search the dense index and rerank results
reranked_results = index.search(
    namespace=index_name,
    query={
        "top_k": 3,
        "inputs": {
            'text': query
        }
    } 
)

for hit in reranked_results['result']['hits']:
    hit_dict = hit.to_dict()
    print(hit_dict)

{'_id': '9781419725456', '_score': 15.36328125, 'fields': {'authors': 'Jeff Kinney', 'bib_id': '133840', 'series': 'Diary of a wimpy kid ;', 'subjects': ['Families', 'Vacations', 'Family life', 'Vacations', 'JUVENILE FICTION Humorous Stories', 'JUVENILE FICTION Comics & Graphic Novels General'], 'synopsis': 'Greg Heffley and his family escape to a tropical island resort for some much-needed rest and relaxation, but sun poisoning, stomach troubles, and venomous creatures all threaten their vacation.', 'text': "The book Diary of a wimpy kid : was written by Jeff Kinney.  Here is a synopsis: Greg Heffley and his family escape to a tropical island resort for some much-needed rest and relaxation, but sun poisoning, stomach troubles, and venomous creatures all threaten their vacation..  It is in the Diary of a wimpy kid ; series and has the following subjects: ['Families', 'Vacations', 'Family life', 'Vacations', 'JUVENILE FICTION Humorous Stories', 'JUVENILE FICTION Comics & Graphic Novels 